# Subtask 1 — U-Net with Ordinal Head (Notebook 3 of 5)

**Goal:** Train a spatial model that uses patch context for pixel-level ordinal regression.

**Key design choice — Ordinal soft labels:**  
For true class k, the target distribution spreads probability mass to adjacent classes:
```
k=0: [0.8, 0.2, 0.0, 0.0, 0.0]
k=1: [0.1, 0.8, 0.1, 0.0, 0.0]
k=2: [0.0, 0.1, 0.8, 0.1, 0.0]
k=3: [0.0, 0.0, 0.1, 0.8, 0.1]
k=4: [0.0, 0.0, 0.0, 0.2, 0.8]
```
This directly rewards Accuracy±1 by training the model to not be confident in the wrong direction.

**HITL gates:**
- After epoch 3: check if loss is decreasing; if stuck → change TEMPORAL_OPTION
- After training: review confusion matrix; if edge classes (0 or 4) have low recall → increase CLASS_WEIGHTS

**TEMPORAL_OPTION:**
- `'concat'` — all 340 channels concatenated (slow, full info)
- `'summary'` — 20 channels: per-band temporal mean + std (fast, good baseline)
- `'seasonal'` — 40 channels: per-band per-season mean (medium)

## 0. Setup

In [ ]:
# !pip install -q rasterio pillow tqdm

import csv
import json
import os
import time
from pathlib import Path
from collections import Counter
from urllib.parse import urljoin
from urllib.request import urlopen

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import pandas as pd

# ── REPO ROOT — works whether JupyterLab cwd is project root or notebooks/ ──
_cwd = Path(os.getcwd())
REPO_ROOT = _cwd if (_cwd / "scripts").exists() else _cwd.parent

# ── CONFIG ────────────────────────────────────────────────────────────────────
DATA_DIR         = None          # local path or None for HF streaming
LABEL_NAME       = "viticulture"
HF_BASE          = "https://huggingface.co/datasets/m-sakka/agripotential/resolve/main/"
CKPT_DIR         = REPO_ROOT / "results" / "subtask1" / "checkpoints"
OUT_DIR          = REPO_ROOT / "results" / "subtask1" / "unet"
for d in [CKPT_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── TEMPORAL OPTION: change based on HITL feedback ───────────────────────────
TEMPORAL_OPTION  = "summary"    # 'concat' | 'summary' | 'seasonal'

# ── TRAINING HYPERPARAMETERS ──────────────────────────────────────────────────
BATCH_SIZE       = 8
LR               = 1e-3
N_EPOCHS         = 30
PATIENCE         = 5
SOFT_LABEL_EPS   = 0.1
CLASS_WEIGHTS    = None         # set to [w0,w1,w2,w3,w4] if edge class recall is low
RANDOM_SEED      = 42
# num_workers=0 is safest on RunPod; increase to 2-4 if you have spare CPUs
NUM_WORKERS      = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"Device: {DEVICE}")
print(f"Repo root: {REPO_ROOT}")
print(f"Temporal option: {TEMPORAL_OPTION}")

## 1. Metadata

In [ ]:
import rasterio
from rasterio.windows import Window

def ref(name):
    if DATA_DIR:
        return str(Path(DATA_DIR) / name)
    return urljoin(HF_BASE, name)

def read_csv(name):
    r = ref(name)
    if r.startswith("http"):
        with urlopen(r, timeout=60) as h:
            lines = [l.decode("utf-8") for l in h]
    else:
        lines = Path(r).read_text().splitlines()
    return list(csv.DictReader(lines))

metadata   = read_csv("metadata.csv")
train_rows = read_csv("train.csv")
val_rows   = read_csv("val.csv")

meta_df = pd.DataFrame(metadata)
meta_df["date"] = pd.to_datetime(meta_df[["year","month","day"]].astype(int))
meta_df = meta_df.sort_values("date").reset_index(drop=True)
sentinel_files = list(meta_df["filename"])
n_times = len(sentinel_files)
n_bands = 10
tf_months = list(meta_df["month"].astype(int))

SPRING_IDX = [i for i,m in enumerate(tf_months) if m in (3,4,5)]
SUMMER_IDX = [i for i,m in enumerate(tf_months) if m in (6,7,8)]
AUTUMN_IDX = [i for i,m in enumerate(tf_months) if m in (9,10,11)]
WINTER_IDX = [i for i,m in enumerate(tf_months) if m in (12,1,2)]

print(f"Timeframes: {n_times}, Train: {len(train_rows)}, Val: {len(val_rows)}")

## 2. Dataset

In [ ]:
def load_patch_tensor(row_idx, col_idx, patch_size, temporal_option):
    """Load and return a feature tensor (C, H, W) for one patch."""
    win = Window(col_idx, row_idx, patch_size, patch_size)
    frames = []
    for fname in sentinel_files:
        with rasterio.open(ref(fname)) as src:
            arr = src.read(window=win).astype(np.float32)  # (B, H, W)
        frames.append(arr)
    stack = np.stack(frames, axis=0)  # (T, B, H, W)

    if temporal_option == "concat":
        # (T*B, H, W)
        out = stack.reshape(n_times * n_bands, patch_size, patch_size)
    elif temporal_option == "summary":
        # (2*B, H, W): per-band mean + std across time
        mean = stack.mean(axis=0)   # (B, H, W)
        std  = stack.std(axis=0)    # (B, H, W)
        out  = np.concatenate([mean, std], axis=0)
    elif temporal_option == "seasonal":
        # (4*B, H, W): per-band per-season mean
        parts = []
        for idx in [SPRING_IDX, SUMMER_IDX, AUTUMN_IDX, WINTER_IDX]:
            if idx:
                parts.append(stack[idx].mean(axis=0))
            else:
                parts.append(np.zeros((n_bands, patch_size, patch_size), dtype=np.float32))
        out = np.concatenate(parts, axis=0)
    else:
        raise ValueError(f"Unknown TEMPORAL_OPTION: {temporal_option}")
    return out  # (C, H, W)


def compute_train_stats(n_patches=100):
    """Compute per-channel mean and std from a sample of training patches."""
    label_ref = ref(f"{LABEL_NAME}.tif")
    samples = []
    for patch in train_rows[:n_patches]:
        r, c, ps = int(patch["row"]), int(patch["col"]), int(patch["patch_size"])
        x = load_patch_tensor(r, c, ps, TEMPORAL_OPTION)  # (C, H, W)
        samples.append(x.reshape(x.shape[0], -1).mean(axis=1))
    means = np.stack(samples).mean(axis=0)  # (C,)
    # Second pass for std
    samples2 = []
    for patch in train_rows[:n_patches]:
        r, c, ps = int(patch["row"]), int(patch["col"]), int(patch["patch_size"])
        x = load_patch_tensor(r, c, ps, TEMPORAL_OPTION)
        samples2.append(((x.reshape(x.shape[0], -1) - means[:, None]) ** 2).mean(axis=1))
    stds = np.sqrt(np.stack(samples2).mean(axis=0)) + 1e-6
    return means, stds

print("Computing normalisation stats from 100 training patches...")
NORM_MEAN, NORM_STD = compute_train_stats(100)
n_channels = len(NORM_MEAN)
print(f"Channels: {n_channels}, mean range: [{NORM_MEAN.min():.1f}, {NORM_MEAN.max():.1f}]")
np.save(CKPT_DIR / f"norm_mean_{TEMPORAL_OPTION}.npy", NORM_MEAN)
np.save(CKPT_DIR / f"norm_std_{TEMPORAL_OPTION}.npy", NORM_STD)

In [ ]:
label_ref = ref(f"{LABEL_NAME}.tif")
train_ds = AgriPotentialDataset(train_rows, label_ref, TEMPORAL_OPTION, NORM_MEAN, NORM_STD, augment=True)
val_ds   = AgriPotentialDataset(val_rows,   label_ref, TEMPORAL_OPTION, NORM_MEAN, NORM_STD, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 3. U-Net Architecture

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class UNet(nn.Module):
    """Lightweight 4-level U-Net for patch-level segmentation."""
    def __init__(self, in_channels, n_classes, base_ch=32):
        super().__init__()
        c = base_ch
        self.enc1 = ConvBlock(in_channels, c)
        self.enc2 = ConvBlock(c,   c*2)
        self.enc3 = ConvBlock(c*2, c*4)
        self.bot  = ConvBlock(c*4, c*8)
        self.up3  = nn.ConvTranspose2d(c*8, c*4, 2, stride=2)
        self.dec3 = ConvBlock(c*8, c*4)
        self.up2  = nn.ConvTranspose2d(c*4, c*2, 2, stride=2)
        self.dec2 = ConvBlock(c*4, c*2)
        self.up1  = nn.ConvTranspose2d(c*2, c, 2, stride=2)
        self.dec1 = ConvBlock(c*2, c)
        self.head = nn.Conv2d(c, n_classes, 1)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        e1 = self.enc1(x)         # (B, c,   H,   W)
        e2 = self.enc2(self.pool(e1))  # (B, c*2, H/2, W/2)
        e3 = self.enc3(self.pool(e2))  # (B, c*4, H/4, W/4)
        b  = self.bot(self.pool(e3))   # (B, c*8, H/8, W/8)
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)  # (B, n_classes, H, W)


model = UNet(in_channels=n_channels, n_classes=5).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"U-Net parameters: {n_params:,}")
print(f"Input channels: {n_channels}  (TEMPORAL_OPTION={TEMPORAL_OPTION})")

## 4. Ordinal Soft-Label Loss

In [ ]:
def build_ordinal_soft_targets(labels, n_classes=5, eps=0.1):
    """
    Convert integer label tensor (B, H, W) to soft ordinal targets (B, n_classes, H, W).
    Adjacent classes get weight eps/2 each; true class gets 1 - eps (or 1 - eps/2 at edges).
    """
    B, H, W = labels.shape
    soft = torch.zeros(B, n_classes, H, W, device=labels.device)
    for k in range(n_classes):
        mask = (labels == k)  # (B, H, W)
        soft[:, k][mask] = 1.0
    # Spread eps to neighbours
    soft_new = soft.clone()
    for k in range(n_classes):
        if k > 0:
            soft_new[:, k-1] += eps * soft[:, k]
        if k < n_classes - 1:
            soft_new[:, k+1] += eps * soft[:, k]
        soft_new[:, k]   -= eps * soft[:, k]  # subtract from centre
    # Re-normalise rows
    soft_new = soft_new / soft_new.sum(dim=1, keepdim=True).clamp(min=1e-9)
    return soft_new  # (B, n_classes, H, W)


def ordinal_soft_loss(logits, labels, eps=SOFT_LABEL_EPS, class_weights=None):
    """Soft cross-entropy loss with ordinal label smoothing."""
    soft = build_ordinal_soft_targets(labels, eps=eps)
    log_probs = F.log_softmax(logits, dim=1)  # (B, C, H, W)
    loss = -(soft * log_probs)  # (B, C, H, W)
    if class_weights is not None:
        w = torch.tensor(class_weights, device=logits.device, dtype=torch.float32)
        # Weight by true class
        w_map = w[labels.long()]  # (B, H, W)
        loss = loss.sum(dim=1) * w_map  # (B, H, W)
    else:
        loss = loss.sum(dim=1)  # (B, H, W)
    return loss.mean()


# Quick sanity check
dummy_lbl = torch.tensor([[[0, 1, 2, 3, 4]]])
soft = build_ordinal_soft_targets(dummy_lbl)
print("Soft targets for classes 0–4 (centre column):")
for k in range(5):
    print(f"  k={k}: {soft[0, :, 0, k].tolist()}")

## 5. Training Loop

In [ ]:
def accuracy_pm1(y_true, y_pred):
    return float(np.mean(np.abs(y_true.astype(int) - y_pred.astype(int)) <= 1))


def validate(model, loader):
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for x, lbl in loader:
            x = x.to(DEVICE)
            logits = model(x)  # (B, 5, H, W)
            pred = logits.argmax(dim=1).cpu().numpy().ravel()
            all_pred.append(pred)
            all_true.append(lbl.numpy().ravel())
    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    return accuracy_pm1(y_true, y_pred), float(np.mean(y_pred == y_true))


optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

class_weights_t = CLASS_WEIGHTS  # None or list of 5 floats

history = []
best_acc_pm1 = 0.0
patience_counter = 0
best_ckpt_path = CKPT_DIR / f"unet_{TEMPORAL_OPTION}_best.pt"

print(f"Training for up to {N_EPOCHS} epochs (early stopping patience={PATIENCE})")
print(f"Best checkpoint → {best_ckpt_path}")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_losses = []
    t0 = time.time()
    for x, lbl in tqdm(train_loader, desc=f"Epoch {epoch}/{N_EPOCHS}", leave=False):
        x, lbl = x.to(DEVICE), lbl.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(x)
            loss   = ordinal_soft_loss(logits, lbl, eps=SOFT_LABEL_EPS, class_weights=class_weights_t)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_losses.append(loss.item())

    scheduler.step()
    train_loss = np.mean(epoch_losses)
    val_pm1, val_exact = validate(model, val_loader)
    elapsed = time.time() - t0

    row = {"epoch": epoch, "train_loss": train_loss, "val_exact": val_exact,
           "val_acc_pm1": val_pm1, "lr": scheduler.get_last_lr()[0]}
    history.append(row)

    improved = val_pm1 > best_acc_pm1
    if improved:
        best_acc_pm1 = val_pm1
        patience_counter = 0
        torch.save({"epoch": epoch, "model_state": model.state_dict(),
                    "val_acc_pm1": val_pm1, "temporal_option": TEMPORAL_OPTION,
                    "n_channels": n_channels, "norm_mean": NORM_MEAN.tolist(),
                    "norm_std": NORM_STD.tolist()}, best_ckpt_path)
        marker = " ← best"
    else:
        patience_counter += 1
        marker = f" ({patience_counter}/{PATIENCE})"

    print(f"Epoch {epoch:3d}  loss={train_loss:.4f}  "
          f"val_exact={val_exact:.4f}  val_acc±1={val_pm1:.4f}  "
          f"[{elapsed:.0f}s]{marker}")

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

# Save history
import pandas as pd
hist_df = pd.DataFrame(history)
hist_df.to_csv(OUT_DIR / f"training_history_{TEMPORAL_OPTION}.csv", index=False)
print(f"\nBest val Acc±1: {best_acc_pm1:.4f}  (saved to {best_ckpt_path})")

## 6. Training Curves

In [ ]:
hist_df = pd.read_csv(OUT_DIR / f"training_history_{TEMPORAL_OPTION}.csv")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_df["epoch"], hist_df["train_loss"], label="Train loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_title("Training loss")

ax2.plot(hist_df["epoch"], hist_df["val_acc_pm1"],  label="Val Acc±1", linewidth=2)
ax2.plot(hist_df["epoch"], hist_df["val_exact"],     label="Val exact", linestyle="--")
ax2.axhline(0.90, color="red", linestyle=":", label="Target 0.90")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_title("Validation accuracy")

plt.suptitle(f"U-Net training — TEMPORAL_OPTION={TEMPORAL_OPTION}", fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / f"training_curves_{TEMPORAL_OPTION}.png", dpi=120)
plt.show()
print(f"Saved → {OUT_DIR}/training_curves_{TEMPORAL_OPTION}.png")

## 7. Post-Training Evaluation & Visual Inspection

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.colors as mcolors

# Load best checkpoint
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Loaded checkpoint from epoch {ckpt['epoch']}  (val Acc±1={ckpt['val_acc_pm1']:.4f})")

# Full validation
all_pred, all_true = [], []
with torch.no_grad():
    for x, lbl in tqdm(val_loader, desc="Final val"):
        x = x.to(DEVICE)
        pred = model(x).argmax(dim=1).cpu().numpy().ravel()
        all_pred.append(pred)
        all_true.append(lbl.numpy().ravel())

y_pred = np.concatenate(all_pred)
y_true = np.concatenate(all_true)
CLASS_NAMES = ["Very Low","Low","Medium","High","Very High"]

cm = confusion_matrix(y_true, y_pred, labels=range(5))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues")
acc_pm1  = accuracy_pm1(y_true, y_pred)
acc_exact = float(np.mean(y_pred == y_true))
ax.set_title(f"U-Net val  exact={acc_exact:.4f}  Acc±1={acc_pm1:.4f}")
plt.tight_layout()
plt.savefig(OUT_DIR / f"confusion_unet_{TEMPORAL_OPTION}.png", dpi=120)
plt.show()

print(f"\nFinal val  exact={acc_exact:.4f}  Acc±1={acc_pm1:.4f}")
if acc_pm1 >= 0.90:
    print("✓ Acc±1 ≥ 0.90 → proceed to Notebook 4 or directly to Notebook 5.")
else:
    print("✗ Below target — try TEMPORAL_OPTION='concat' or increase CLASS_WEIGHTS for edge classes.")

In [ ]:
# Visual comparison on 5 val patches
from PIL import Image

VIS_DIR = REPO_ROOT / "results" / "subtask1" / "val_vis_unet"
VIS_DIR.mkdir(parents=True, exist_ok=True)
CLASS_COLORS = ["#d73027","#fc8d59","#fee08b","#91cf60","#1a9850"]
cmap = mcolors.ListedColormap(CLASS_COLORS)
norm = mcolors.BoundaryNorm([-0.5,0.5,1.5,2.5,3.5,4.5], cmap.N)

model.eval()
fig, axes = plt.subplots(5, 2, figsize=(8, 15))

with rasterio.open(label_ref) as label_src:
    for row_i, patch in enumerate(val_rows[:5]):
        r, c, ps = int(patch["row"]), int(patch["col"]), int(patch["patch_size"])
        lbl = label_src.read(1, window=Window(c, r, ps, ps)).astype(np.uint8)
        x_t = val_ds[row_i][0].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            pred = model(x_t).argmax(dim=1)[0].cpu().numpy().astype(np.uint8)
        apm1 = accuracy_pm1(lbl.ravel(), pred.ravel())
        axes[row_i, 0].imshow(lbl,  cmap=cmap, norm=norm, interpolation="nearest")
        axes[row_i, 0].set_title(f"GT  {patch['patch_id']}", fontsize=8)
        axes[row_i, 0].axis("off")
        axes[row_i, 1].imshow(pred, cmap=cmap, norm=norm, interpolation="nearest")
        axes[row_i, 1].set_title(f"Pred  Acc±1={apm1:.3f}", fontsize=8)
        axes[row_i, 1].axis("off")
        Image.fromarray(pred).save(VIS_DIR / f"{patch['patch_id']}_pred.png")

plt.suptitle(f"U-Net val predictions ({TEMPORAL_OPTION})", fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / f"val_vis_{TEMPORAL_OPTION}.png", dpi=120)
plt.show()
print(f"Saved → {OUT_DIR}/val_vis_{TEMPORAL_OPTION}.png")

## 8. HITL Handoff

After reviewing the training curves and visual predictions:

1. **Is val Acc±1 better than the pixel baseline from Notebook 2?** If not, try `TEMPORAL_OPTION='concat'` for a richer feature set.
2. **Are predictions spatially smooth?** If noisy → Notebook 4's 3×3 median filter will help.
3. **Which classes are confused?** If classes 0 or 4 (extreme ends) are missed → increase their weight in `CLASS_WEIGHTS = [2.0, 1.0, 1.0, 1.0, 2.0]`.
4. **Is the U-Net better than the pixel baseline?** → Ensemble in Notebook 4 for best of both.